In [1]:
%load_ext autoreload
%autoreload 2

import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent


import data_loader as data_loader
import visualization as viz
import analysis_features as feats


RAW_DIR = PROJECT_ROOT / "Data"      # raw data
PROCESSED_DIR = "Processed_data" # output
FS = 50                        # sampling rate

print(f"Current Working Directory: {Path.cwd()}")
print(f"Project Root Detected:     {PROJECT_ROOT}")
print(f"Raw data path:             {RAW_DIR}")

os.makedirs(PROCESSED_DIR, exist_ok=True)

print(":) All set")

Current Working Directory: /Users/louinsleung/EEN210/Xiaoye_week3_code
Project Root Detected:     /Users/louinsleung/EEN210
Raw data path:             /Users/louinsleung/EEN210/Data
:) All set


In [2]:
# batch processing all csv files to parquet
print(f"batch processing from '{RAW_DIR}'.")

data_loader.batch_process(RAW_DIR, PROCESSED_DIR)

print("：）Done!")


batch processing from '/Users/louinsleung/EEN210/Data'.
Found 119 CSV files in /Users/louinsleung/EEN210/Data
[Skip] FallingForwards_p1_1_35sek_20260206_113359.parquet already exists.
[Skip] FallingForwards_p1_1_35sek_20260206_113359.csv already exists.
[Skip] FallingForwards_p4_1_35sek_20260206_115209.parquet already exists.
[Skip] FallingForwards_p4_1_35sek_20260206_115209.csv already exists.
[Skip] FallingForwards_p4_2_35sek_20260206_115310.parquet already exists.
[Skip] FallingForwards_p4_2_35sek_20260206_115310.csv already exists.
[Skip] FallingForwards_p4_3_35sek_20260206_115411.parquet already exists.
[Skip] FallingForwards_p4_3_35sek_20260206_115411.csv already exists.
[Skip] Fallingbackwards_p1_1_35sek_20260205_113403.parquet already exists.
[Skip] Fallingbackwards_p1_1_35sek_20260205_113403.csv already exists.
[Skip] Fallingbackwards_p1_2_35sek_20260205_113455.parquet already exists.
[Skip] Fallingbackwards_p1_2_35sek_20260205_113455.csv already exists.
[Skip] Fallingbackward

In [ ]:
# raw data visualization

targets = {
    "Fall (Backwards)": "Fallingbackwards_p4_1",  
    "Cyclic ADL (Walking)": "Walking_p4_1_35sek",      
    "Sit-to-Stand": "SittingToStanding"     
}


all_files = sorted(glob.glob(os.path.join(PROCESSED_DIR, "long", "*.parquet")))
data_dict = {}


print("Loading representative files...")
for label, keyword in targets.items():
    matches = [f for f in all_files if keyword.lower() in os.path.basename(f).lower()]
    if matches:
        data_dict[label] = data_loader.load_long_parquet(matches[0])
        print(f"Loaded {label}")
    else:
        print(f" Warning: File not found for keyword '{keyword}'")


if len(data_dict) > 0:
    print("\n=== Figure 1: Time Series Trends (Line Plots) ===")
    viz.plot_raw_lines(data_dict, feature="acc_mag")
    
    print("\n=== Figure 2: Data Distribution (Histograms) ===")
    viz.plot_raw_hists(data_dict, feature="acc_mag")
    
    print("\n=== Figure 3: Feature Relationships (Scatter Plots) ===")
    viz.plot_raw_scatter(data_dict, x_col="acc_mag", y_col="gyro_mag")
else:
    print("No data loaded.")
  


Loading representative files...
Loaded Fall (Backwards)
Loaded Cyclic ADL (Walking)
Loaded Sit-to-Stand

=== Figure 1: Time Series Trends (Line Plots) ===


invalid command name "4621558144process_stream_events"
    while executing
"4621558144process_stream_events"
    ("after" script)



=== Figure 2: Data Distribution (Histograms) ===

=== Figure 3: Feature Relationships (Scatter Plots) ===


In [ ]:

# load all processed parquet files 
all_parquet_paths = sorted(glob.glob(os.path.join(PROCESSED_DIR, "long", "*.parquet")))
print(f"Loading {len(all_parquet_paths)} files for feature extraction...")

data_cache = {}
for p in all_parquet_paths:
    # Use basename as key
    fname = os.path.basename(p)
    try:
        data_cache[fname] = data_loader.load_long_parquet(p)
    except Exception as e:
        print(f"Error loading {fname}: {e}")

# Run Feature Extraction Pipeline
print("Starting feature extraction (Window=2.0s, Step=1.0s)...")
# ?? Window size = 2.0s is chosen to capture a fall event 
# ?? Step size = 1.0s
feature_df = feats.build_feature_dataset(data_cache, window_s=2.0, step_s=1.0, fs=50)

print(f"Extraction complete. Generated {len(feature_df)} windows.")
print("Class distribution:")
print(feature_df["activity"].value_counts())

feature_df.head()

Loading 117 files for feature extraction...
Starting feature extraction (Window=2.0s, Step=1.0s)...
Extraction complete. Generated 2732 windows.
Class distribution:
activity
Sit-Stand    1402
Fall          708
Walking       622
Name: count, dtype: int64


,acc_max,acc_mean,acc_std,gyro_max,gyro_mean,gyro_std,acc_skew,acc_kurtosis,acc_energy,gyro_energy,file,label,activity,t_start,t_end
0,1.018086,0.983577,0.018482,24.044311,11.261983,4.619531,0.013947,-0.763194,0.967766,148.172327,FallingForwards_p1_1_35sek_20260206_113359.par...,1,Fall,0.591289,2.564783
1,1.003195,0.976370,0.010726,13.005480,7.143284,3.133417,-0.320820,-0.106987,0.953414,60.844809,FallingForwards_p1_1_35sek_20260206_113359.par...,1,Fall,1.589988,3.564779
2,1.146386,0.977631,0.033575,26.033567,7.093344,4.217540,-0.792816,15.881548,0.956889,68.103176,FallingForwards_p1_1_35sek_20260206_113359.par...,1,Fall,2.591526,4.564791
3,1.146386,0.978916,0.042297,26.033567,7.771516,4.148205,-0.044857,6.042578,0.960065,77.604065,FallingForwards_p1_1_35sek_20260206_113359.par...,1,Fall,3.589703,5.564791
4,1.097998,0.978169,0.033840,18.621109,6.992108,3.356218,1.036305,1.819987,0.957959,60.153778,FallingForwards_p1_1_35sek_20260206_113359.par...,1,Fall,4.591483,6.564818


In [ ]:

# check max acceleration
viz.plot_boxplot_stat(feature_df, feature="acc_max", label_col="activity")

#check Gyroscope Std
viz.plot_boxplot_stat(feature_df, feature="gyro_std", label_col="activity")

# check Kurtosis
viz.plot_boxplot_stat(feature_df, feature="acc_kurtosis", label_col="activity")


/Users/louinsleung/EEN210/Xiaoye_week3_code/visualization.py:167: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x=label_col, y=feature, data=df, palette="Set2")
/Users/louinsleung/EEN210/Xiaoye_week3_code/visualization.py:167: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x=label_col, y=feature, data=df, palette="Set2")
/Users/louinsleung/EEN210/Xiaoye_week3_code/visualization.py:167: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x=label_col, y=feature, data=df, palette="Set2")


In [ ]:

# binary labels, fall vs non-fall
# new dataframe to avoid modifying the original one
binary_df = feature_df.copy()

# If it's 'Fall', keep it. Everything else becomes 'Non-Fall'.
binary_df["binary_class"] = binary_df["activity"].apply(
    lambda x: "Fall" if x == "Fall" else "Non-Fall"
)

# distribution to check balance
print("Binary Class Distribution:")
print(binary_df["binary_class"].value_counts())

# Visualize in 3D
# Selected features based on previous boxplot analysis:
# - acc_max: Represents Impact Intensity
# - gyro_std: Represents Rotational Violence
# - acc_kurtosis: Represents Signal Impulsiveness (Spikiness)

print("\nPlotting 3D Feature Space (Fall vs Non-Fall)...")
viz.plot_3d_feature_space(
    binary_df, 
    x="acc_max", 
    y="gyro_std", 
    z="acc_kurtosis", 
    label_col="binary_class"
)

Binary Class Distribution:
binary_class
Non-Fall    2024
Fall         708
Name: count, dtype: int64

Plotting 3D Feature Space (Fall vs Non-Fall)...


invalid command name "4621252416process_stream_events"
    while executing
"4621252416process_stream_events"
    ("after" script)


In [ ]:
if "binary_df" not in locals():
    binary_df = feature_df.copy()
    binary_df["binary_class"] = binary_df["activity"].apply(lambda x: "Fall" if x == "Fall" else "Non-Fall")


# features to analyze based on boxplot insights and domain knowledge
selected_features = [
    "acc_max",       # Impact intensity
    "gyro_std",      # Rotational intensity
    "acc_kurtosis",  # Signal impulsiveness (spikiness)
    "acc_energy",    # Total energy
    "acc_skew"       # Asymmetry of distribution
]


# result
stats_report = feats.get_statistical_report(
    binary_df, 
    selected_features, 
    group_col="binary_class", 
    target_label="Fall"
)


display(stats_report)

,Feature,Fall (Mean +/- Std),Non-Fall (Mean +/- Std),t-statistic,p-value,Significance
0,acc_max,1.12 +/- 0.35,1.21 +/- 0.28,-6.22,7.31e-10,***
1,gyro_std,8.92 +/- 13.60,13.97 +/- 16.09,-8.10,1.16e-15,***
2,acc_kurtosis,2.49 +/- 4.26,1.99 +/- 3.72,2.79,5.44e-03,**
3,acc_energy,0.94 +/- 0.08,0.98 +/- 0.07,-12.51,8.53e-34,***
4,acc_skew,0.19 +/- 0.97,0.40 +/- 0.87,-4.98,7.23e-07,***


In [ ]:
import analysis_features as feats


filtered_df = binary_df.copy()

silent_fall_indices = filtered_df[
    (filtered_df["binary_class"] == "Fall") & 
    (filtered_df["acc_max"] < 1.5)
].index


active_stats_df = filtered_df.drop(silent_fall_indices)

print(f"Original Windows: {len(binary_df)}")
print(f"Active Windows (High Impact Falls + All Non-Falls): {len(active_stats_df)}")



refined_report = feats.get_statistical_report(
    active_stats_df, 
    selected_features, 
    group_col="binary_class", 
    target_label="Fall"
)

display(refined_report)

Original Windows: 2732
Active Windows (High Impact Falls + All Non-Falls): 2069

Generating REFINED Statistical Report (Active Events Only)...


,Feature,Fall (Mean +/- Std),Non-Fall (Mean +/- Std),t-statistic,p-value,Significance
0,acc_max,2.33 +/- 0.49,1.21 +/- 0.28,15.21,2.86e-19,***
1,gyro_std,48.63 +/- 13.40,13.97 +/- 16.09,17.07,9.47e-22,***
2,acc_kurtosis,7.61 +/- 6.38,1.99 +/- 3.72,5.89,4.68e-07,***
3,acc_energy,1.08 +/- 0.10,0.98 +/- 0.07,6.42,7.53e-08,***
4,acc_skew,2.18 +/- 0.95,0.40 +/- 0.87,12.55,2.07e-16,***


In [ ]:

# test on fall
test_file_label = "Fall (Backwards)"

if 'data_dict' in locals() and test_file_label in data_dict:
    df_test = data_dict[test_file_label]
else:
    df_test = data_loader.load_long_parquet("Xiaoye_week3_code/Processed_data/long/Fallingbackwards_p4_2_35sek_20260205_112600.parquet")

# parameters for event detection
# threshold = 0.2 

detected_events = feats.detect_events(
    df_test, 
    signal_col="acc_mag", 
    fs=50, 
    window_s=0.5,   
    threshold=0.2,  
    merge_gap_s=1.5 # gap less than 1.5s is considered the same event
)

print(f"Detected {len(detected_events)} events:")
for i, (s, e) in enumerate(detected_events):
    print(f"  Event {i+1}: {s:.2f}s -> {e:.2f}s (Duration: {e-s:.2f}s)")


viz.plot_segmentation(df_test, detected_events, title=f"Segmentation Result: {test_file_label}")

Detected 1 events:
  Event 1: 17.26s -> 18.48s (Duration: 1.22s)


invalid command name "4621531136process_stream_events"
    while executing
"4621531136process_stream_events"
    ("after" script)


In [ ]:
import preprocessing as pre



if 'binary_df' not in locals():
    print("Please run previous cells to generate binary_df first.")
else:
   
    clean_df = pre.clean_feature_matrix(binary_df)
    

    feature_columns = [
        "acc_max", "acc_mean", "acc_std", "acc_skew", "acc_kurtosis", "acc_energy",
        "gyro_max", "gyro_mean", "gyro_std", "gyro_energy"
    ]
    

    valid_features = [c for c in feature_columns if c in clean_df.columns]
    
    # Normalization z-score
    final_dataset, scaler = pre.normalize_features(clean_df, valid_features, method='z-score')
    

    print("\nOriginal Means (acc_max):", binary_df["acc_max"].mean())
    print("Scaled Means (acc_max):  ", round(final_dataset["acc_max"].mean(), 2), "(Should be close to 0)")
    
    display(final_dataset.head())
    
    final_dataset.to_csv(os.path.join(PROCESSED_DIR, "final_dataset_week3.csv"), index=False)
    print(f"Dataset saved to {os.path.join(PROCESSED_DIR, 'final_dataset_week3.csv')}")

[Preprocessing] Features normalized using z-score.

Original Means (acc_max): 1.1862267902904051
Scaled Means (acc_max):   0.0 (Should be close to 0)

Final Dataset Ready for Machine Learning!


,acc_max,acc_mean,acc_std,gyro_max,gyro_mean,gyro_std,acc_skew,acc_kurtosis,acc_energy,gyro_energy,file,label,activity,t_start,t_end,binary_class
0,-0.558548,0.110027,-0.551186,-0.521075,-0.444704,-0.514550,-0.365974,-0.742848,-0.047876,-0.455074,FallingForwards_p1_1_35sek_20260206_113359.par...,1,Fall,0.591289,2.564783,Fall
1,-0.608017,-0.109484,-0.650260,-0.703562,-0.619250,-0.609596,-0.736809,-0.573585,-0.240256,-0.485957,FallingForwards_p1_1_35sek_20260206_113359.par...,1,Fall,1.589988,3.564779,Fall
2,-0.132349,-0.071096,-0.358416,-0.488189,-0.621366,-0.540260,-1.259659,3.550513,-0.193676,-0.483390,FallingForwards_p1_1_35sek_20260206_113359.par...,1,Fall,2.591526,4.564791,Fall
3,-0.132349,-0.031956,-0.247009,-0.488189,-0.592626,-0.544694,-0.431113,1.012639,-0.151104,-0.480031,FallingForwards_p1_1_35sek_20260206_113359.par...,1,Fall,3.589703,5.564791,Fall
4,-0.293088,-0.054711,-0.355030,-0.610728,-0.625657,-0.595347,0.766535,-0.076540,-0.179333,-0.486202,FallingForwards_p1_1_35sek_20260206_113359.par...,1,Fall,4.591483,6.564818,Fall


Dataset saved to Processed_data/final_dataset_week3.csv
